# Fabula — Unsloth QLoRA Fine-Tuning & GGUF Export (Task D-02)

This notebook fine-tunes `Qwen/Qwen2.5-3B-Instruct` in 4-bit on the Fabula 4-stage SFT mystery dataset (`train.jsonl`, 2,048 samples) using **Unsloth** on Kaggle's free Tesla T4 GPU.

### Optimizations Applied:
1. **Dynamic Path Resolution:** Automatically locates `train.jsonl` anywhere under `/kaggle/input/` or `/kaggle/working/`.
2. **2560 Context Window:** Fits 99% of samples with optimal attention throughput on Tesla T4.
3. **Response-Only Masking (`train_on_responses_only`):** System & user prompt tokens are masked with `-100`; cross-entropy loss is computed **solely on assistant JSON completions**.
4. **Step Checkpointing & Auto-Resume:** Automatically detects previous checkpoints in case of interruption.
5. **Quantized GGUF Export:** Merges weights to 16-bit and exports to `Q4_K_M` GGUF for local CPU execution.

In [ ]:
# Step 1: Install Unsloth and dependencies
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes
print("Unsloth installed successfully!")

In [ ]:
# Step 2: Load Model & Tokenizer in 4-bit
import unsloth
from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only

MAX_SEQ_LENGTH = 2560
MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"

print(f"Loading {MODEL_NAME}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,  # Auto-select Float16 for Tesla T4
    load_in_4bit=True,
)

# Step 3: Attach Parameter-Efficient LoRA Adapters (Rank 16)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print("Model loaded and LoRA adapters attached!")

In [ ]:
# Step 4: Dynamic Dataset Path Resolution & Formatting
import json
import os
from datasets import Dataset

def find_file(filename: str) -> str:
    candidates = [
        os.path.join("/kaggle/working", filename),
        os.path.join("data", "processed", filename),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    if os.path.exists("/kaggle/input"):
        for root, _, files in os.walk("/kaggle/input"):
            if filename in files:
                return os.path.join(root, filename)
    raise FileNotFoundError(f"Could not find '{filename}' in /kaggle/input or /kaggle/working.")

train_path = find_file("train.jsonl")
print(f"Training data located at: {train_path}")

def load_jsonl_messages(filepath):
    data = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                item = json.loads(line)
                formatted_text = tokenizer.apply_chat_template(
                    item["messages"],
                    tokenize=False,
                    add_generation_prompt=False
                )
                data.append({"text": formatted_text})
    return Dataset.from_list(data)

print("Formatting dataset into ChatML structure...")
train_dataset = load_jsonl_messages(train_path)
print(f"Train samples: {len(train_dataset)}")

In [ ]:
# Step 5: SFTTrainer Setup with Step Checkpointing
import glob
from trl import SFTTrainer
from transformers import TrainingArguments

OUTPUT_DIR = "/kaggle/working/fabula_model_output"

trainer_kwargs = {
    "model": model,
    "train_dataset": train_dataset,
    "dataset_text_field": "text",
    "max_seq_length": MAX_SEQ_LENGTH,
    "dataset_num_proc": 2,
    "packing": False,
    "args": TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=15,
        num_train_epochs=2,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=25,
        eval_strategy="no",
        save_strategy="steps",
        save_steps=50,
        save_total_limit=2,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir=OUTPUT_DIR,
        report_to="none",
    ),
}

try:
    trainer = SFTTrainer(processing_class=tokenizer, **trainer_kwargs)
except TypeError:
    trainer = SFTTrainer(tokenizer=tokenizer, **trainer_kwargs)

# Apply response-only loss masking
print("Applying response-only loss masking...")
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

# Check for existing checkpoint to resume from
checkpoints = glob.glob(f"{OUTPUT_DIR}/checkpoint-*") if os.path.exists(OUTPUT_DIR) else []
resume_checkpoint = None
if checkpoints:
    checkpoints = sorted(checkpoints, key=lambda x: int(x.split("-")[-1]) if x.split("-")[-1].isdigit() else 0)
    resume_checkpoint = checkpoints[-1]
    print(f"--> Found existing checkpoint! Resuming from {resume_checkpoint}...")
else:
    print("--> Starting fresh training (512 steps)...")

trainer.train(resume_from_checkpoint=resume_checkpoint)
print("Training Finished successfully!")

In [ ]:
# Step 6: Save LoRA Adapters and Export GGUF Q4_K_M
ADAPTER_DIR = "/kaggle/working/fabula_qlora_adapter"
GGUF_DIR = "/kaggle/working/fabula_gguf"

print(f"Saving fine-tuned LoRA adapter to {ADAPTER_DIR}...")
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("LoRA adapter safely persisted!")

print(f"Exporting merged GGUF (Q4_K_M) to {GGUF_DIR}...")
try:
    model.save_pretrained_gguf(
        GGUF_DIR,
        tokenizer,
        quantization_method="q4_k_m"
    )
    print("GGUF Export Complete! Download Qwen2.5-3B-Instruct.Q4_K_M.gguf from the output tab.")
except Exception as e:
    print(f"GGUF direct export encountered an exception: {e}")
    print("Note: Your LoRA adapter is safely saved in fabula_qlora_adapter and can be converted to GGUF anytime.")